In [1]:
#    نحدد اسم الملف  الذي بنشتغل عليه
# ونحدد اسم ملف الناتج النهائي (نسخة جديدة)     

import pandas as pd

input_file = "Legal_Consultations_Standardized.xlsx"
output_file_clean = "Legal_Consultations_CleanText.xlsx"

print("Input:", input_file)
print("Output:", output_file_clean)


Input: Legal_Consultations_Standardized.xlsx
Output: Legal_Consultations_CleanText.xlsx


In [4]:
#    نقرأ كل الشيتات من ملف الإكسل 
#  نجمعهم في جدول واحد مع عمود يوضح المصدر  (اسم الشيت)

all_sheets = pd.read_excel(input_file, sheet_name=None)

dfs = []
for sheet_name, sheet_df in all_sheets.items():
    if sheet_df is None or sheet_df.empty:
        continue
    temp = sheet_df.copy()
    temp["source"] = sheet_name
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Total rows:", df.shape[0])
print("Columns:", list(df.columns))
df.head(8)


Total rows: 2321
Columns: ['نص الاستشارة', 'التصنيف', 'source']


,نص الاستشارة,التصنيف,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,قضايا الأحوال الشخصية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini


In [6]:
#     نكتشف عمود النص وعمود التصنيف تلقائيًا

TEXT_COL_CANDIDATES = ["نص الاستشارة", "نص_الاستشارة", "text", "case_text"]
LABEL_COL_CANDIDATES = ["التصنيف", "التصنيف_النهائي", "label", "class"]

TEXT_COL = next((c for c in TEXT_COL_CANDIDATES if c in df.columns), None)
LABEL_COL = next((c for c in LABEL_COL_CANDIDATES if c in df.columns), None)

print("Detected TEXT_COL:", TEXT_COL)
print("Detected LABEL_COL:", LABEL_COL)

df[[TEXT_COL, LABEL_COL]].head(8)


Detected TEXT_COL: نص الاستشارة
Detected LABEL_COL: التصنيف


,نص الاستشارة,التصنيف
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,قضايا الأحوال الشخصية
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية


In [7]:
#     تنظيف  للنص
# - حذف الروابط والايميلات
# - حذف الأرقام والرموز والإنجليزي
# - إزالة التشكيل
# - توحيد الحروف (أ/إ/آ -> ا) (ة -> ه) (ى -> ي)
# - تقليل المد والتكرار
# - تنظيف المسافات

import re

AR_DIACRITICS = r"[ًٌٍَُِّْـ]"
TATWEEL = "ـ"

def basic_normalize_ar(text: str) -> str:
    text = "" if pd.isna(text) else str(text)

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(AR_DIACRITICS, "", text)
    text = text.replace(TATWEEL, "")

    text = (text
            .replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
            .replace("ى", "ي")
            .replace("ة", "ه")
            .replace("ؤ", "و").replace("ئ", "ي"))

    text = re.sub(r"[A-Za-z]", " ", text)
    text = re.sub(r"\d+", " ", text)

    text = re.sub(r"[؟،؛…!\"'#\$%&\(\)\*\+,\-\.\/:;<=>@\[\]\\\^_`{\|}~]", " ", text)

    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean_1"] = df[TEXT_COL].apply(basic_normalize_ar)
df[[TEXT_COL, "text_clean_1"]].head(8)


,نص الاستشارة,text_clean_1
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله من الموسسه بدون فواتير وش الحل
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,زوجي هجر البيت ولا يصرف علي العيال من شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,تم استبعادي من مسابقه وظيفيه حكوميه رغم انطباق...
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,البنك سحب مبلغ اكبر من القسط الشهري المتفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...


In [10]:
#    نحذف الكلمات العامة جدًا (من، في، على...)
# ونحذف كلمات قانونية عامة (مثل: قضيه، شكوى، استفسار...)

AR_STOPWORDS = set("""
من في على علي الى إلى عن مع بين عند لدى عندي لدي له لها لهم هن هو هي انا انت انتي نحن هم هذا هذه ذلك تلك
كان تكون يكون كانت كنت جدا فقط ايضا ثم حيث اذا لأن لان قد لقد لا ولا لم لن ما ماذا كيف ليش وش
 مرة مرات قبل بعد خلال حول حتى بدون فوق تحت داخل خارج السلام عليكم ورحمه الله وبركاته 
""".split())

LEGAL_GENERIC = set("""
قضيه قضية قضايا دعوى شكوى شكاوى استفسار سؤال سوال مطلوب ابي ابغى احتاج عندي لدي
  محكمه محكمة القاضي قاضي جلسه جلسة حكم صك صكوك نظام لائحه لائحة ماده مادة ماده اريد
""".split())

def remove_stopwords(text: str) -> str:
    tokens = text.split()
    tokens = [w for w in tokens if (w not in AR_STOPWORDS) and (w not in LEGAL_GENERIC) and (len(w) > 2)]
    return " ".join(tokens)

df["text_clean_2"] = df["text_clean_1"].apply(remove_stopwords)
df[["text_clean_1", "text_clean_2"]].head(8)


,text_clean_1,text_clean_2
0,شريكي سحب سيوله من الموسسه بدون فواتير وش الحل,شريكي سحب سيوله الموسسه فواتير الحل
1,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...,صاحب العمل فصلني سابق انذار اعطاني مكافاه
2,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب,تعرضت لابتزاز بصور خاصه حساب وهمي سناب
3,زوجي هجر البيت ولا يصرف علي العيال من شهور,زوجي هجر البيت يصرف العيال شهور
4,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...,شريت شقه وطلعت فيها عيوب السباكه والمالك يرفض ...
5,تم استبعادي من مسابقه وظيفيه حكوميه رغم انطباق...,استبعادي مسابقه وظيفيه حكوميه رغم انطباق الشروط
6,البنك سحب مبلغ اكبر من القسط الشهري المتفق عليه,البنك سحب مبلغ اكبر القسط الشهري المتفق عليه
7,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي المزرعه


In [12]:
# 1) نوسع قاموس الاستبدالات لكلمات لهجية وأخطاء شائعة
# 2) نسوي "تجذير خفيف" بسيط: نشيل (الـ) وبعض اللواحق الشائعة
# : نوحد شكل الكلمات بدون تخريب المعنى مثل التجذير القاسي

import re

REPLACEMENTS = {
    # رغبات/أسئلة لهجية
    "ابي": "اريد",
    "ابغى": "اريد",
    "ودي": "اريد",
    "وش": "ما",
    "ايش": "ما",
    "ليه": "لماذا",
    "ليش": "لماذا",

    # كلمات شائعة بالاستشارات
    "زوجي": "زوج",
    "زوجتي": "زوجة",
    "ابوي": "اب",
    "امي": "ام",
    "اخوي": "اخ",
    "اختي": "اخت",
    "عيالي": "ابناء",
    "اولادي": "ابناء",

    # مصطلحات قانونية متكررة
    "النفقة": "نفقة",
    "نفقتي": "نفقة",
    "حضانتي": "حضانة",
    "حضانه": "حضانة",
    "طلاقه": "طلاق",
    "الطلاق": "طلاق",
    "زواجي": "زواج",
    "الزواج": "زواج",

    "الميراث": "ميراث",
    "ورث": "ميراث",
    "ورثة": "ورثة",

    "راتبي": "راتب",
    "رواتبي": "راتب",
    "مستحقاتي": "مستحقات",
    "مستحقاتيّ": "مستحقات",

    # شائع في العمل
    "فصلني": "فصل",
    "فصلوني": "فصل",
    "راتبي": "راتب",
    "مكافاتي": "مكافاة",
}


def light_stem_ar(word: str) -> str:
    # نشيل "ال" التعريف إذا كانت الكلمة طويلة بما يكفي
    if word.startswith("ال") and len(word) > 4:
        word = word[2:]


    return word

def normalize_variants(text: str) -> str:
    tokens = text.split()
    out = []
    for w in tokens:
        # استبدال مباشر
        w = REPLACEMENTS.get(w, w)

        # تطبيع خفيف
        w = light_stem_ar(w)

        out.append(w)

    return " ".join(out)

df["final_text"] = df["text_clean_2"].astype(str).apply(normalize_variants)

df[[TEXT_COL, "final_text"]].head(8)


,نص الاستشارة,final_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله موسسه فواتير الحل
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب عمل فصل سابق انذار اعطاني مكافاه
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,تعرضت لابتزاز بصور خاصه حساب وهمي سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,زوج هجر بيت يصرف عيال شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب سباكه والمالك يرفض اصلاح
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,استبعادي مسابقه وظيفيه حكوميه رغم انطباق شروط
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,بنك سحب مبلغ اكبر قسط شهري متفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,اخ كبير رافض يوزع ميراث اب ومستولي مزرعه


In [13]:
#  نشوف النصوص اللي صارت قصيرة جدًا بعد التنظيف
# عشان ننتبه إذا فيه حذف زيادة

df["final_len"] = df["final_text"].apply(lambda x: len(str(x).split()))
print("Final text length stats:")
print(df["final_len"].describe())

print("\nExamples of very short cleaned texts:")
df[df["final_len"] <= 3][[TEXT_COL, "final_text", LABEL_COL]].head(10)


Final text length stats:
count    2321.000000
mean       16.348557
std        18.390578
min         3.000000
25%         7.000000
50%        12.000000
75%        16.000000
max       191.000000
Name: final_len, dtype: float64

Examples of very short cleaned texts:


,نص الاستشارة,final_text,التصنيف
333,نظام الإجازات الميدانية للموظفين.,اجازات ميدانيه للموظفين,القضايا العمالية
405,نظام العمل عن بعد للموظفات وحقوقهن.,عمل للموظفات وحقوقهن,القضايا العمالية


In [14]:
#    نحفظ نسخة جديدة من الإكسل
# ونضمن أن كل شيت يحتفظ بصفوفه، ويضاف له عمود final_text

with pd.ExcelWriter(output_file_clean, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        if sheet_df is None or sheet_df.empty:
            continue

        temp = sheet_df.copy()

        # نربط التنظيف على مستوى الجدول الموحّد عبر index الأصلي
        # لذلك نسوي merge باستخدام الأعمدة الأساسية لو موجودة
        # أبسط حل: نعيد تنظيف نص الشيت نفسه مباشرة هنا
        if TEXT_COL in temp.columns:
            temp["final_text"] = temp[TEXT_COL].apply(basic_normalize_ar).apply(remove_stopwords).apply(normalize_variants)

        temp.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved cleaned academic file:", output_file_clean)


Saved cleaned academic file: Legal_Consultations_CleanText.xlsx


In [15]:
#    نتأكد أن الملف الجديد موجود ومقروء
# ونطبع 8 صفوف من شيت Gemini للتأكد

check = pd.read_excel(output_file_clean, sheet_name=None)
print("New file sheets:", list(check.keys()))

check["Gemini"][[TEXT_COL, LABEL_COL, "final_text"]].head(8)


New file sheets: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


,نص الاستشارة,التصنيف,final_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,شريكي سحب سيوله موسسه فواتير الحل
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,صاحب عمل فصل سابق انذار اعطاني مكافاه
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,تعرضت لابتزاز بصور خاصه حساب وهمي سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,قضايا الأحوال الشخصية,زوج هجر بيت يصرف عيال شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,شريت شقه وطلعت فيها عيوب سباكه والمالك يرفض اصلاح
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,استبعادي مسابقه وظيفيه حكوميه رغم انطباق شروط
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,بنك سحب مبلغ اكبر قسط شهري متفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,اخ كبير رافض يوزع ميراث اب ومستولي مزرعه


In [16]:
# نتأكد أن النص المنظف موجود والتصنيف موجود

print("Columns:", df.columns)

print("\nعدد الصفوف:", df.shape[0])

print("\nالتصنيفات الموجودة:")
print(df["التصنيف"].value_counts())


Columns: Index(['نص الاستشارة', 'التصنيف', 'source', 'text_clean_1', 'text_clean_2',
       'final_text', 'final_len'],
      dtype='str')

عدد الصفوف: 2321

التصنيفات الموجودة:
التصنيف
القضايا العمالية         458
قضايا الأحوال الشخصية    437
القضايا العقارية         327
القضايا الإدارية         291
القضايا التجارية         278
القضايا الجنائية         271
القضايا المالية          259
Name: count, dtype: int64


In [17]:
# تقسيم البيانات للتدريب والاختبار

from sklearn.model_selection import train_test_split

X_text = df["final_text"].fillna("").astype(str)
y_text = df["التصنيف"].astype(str)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text,
    y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))


Train size: 1856
Test size: 465


In [18]:
# تدريب مودل SVM باستخدام TF-IDF

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3,5),
        max_features=20000,
        min_df=2,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(C=0.5, class_weight="balanced"))
])

baseline_model.fit(X_train_text, y_train_text)

print("Model trained successfully.")


Model trained successfully.


In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

y_pred = baseline_model.predict(X_test_text)

print("Accuracy:", accuracy_score(y_test_text, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test_text, y_pred))

labels = sorted(y_test_text.unique())
cm = confusion_matrix(y_test_text, y_pred, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print("\nConfusion Matrix:")
cm_df


Accuracy: 0.9182795698924732

Classification Report:

                       precision    recall  f1-score   support

     القضايا الإدارية       0.86      0.86      0.86        58
     القضايا التجارية       0.84      0.86      0.85        56
     القضايا الجنائية       0.95      0.96      0.95        54
     القضايا العقارية       0.90      1.00      0.95        65
     القضايا العمالية       0.98      0.97      0.97        92
      القضايا المالية       0.93      0.83      0.88        52
قضايا الأحوال الشخصية       0.93      0.91      0.92        88

             accuracy                           0.92       465
            macro avg       0.91      0.91      0.91       465
         weighted avg       0.92      0.92      0.92       465


Confusion Matrix:


,القضايا الإدارية,القضايا التجارية,القضايا الجنائية,القضايا العقارية,القضايا العمالية,القضايا المالية,قضايا الأحوال الشخصية
القضايا الإدارية,50,3,2,2,0,1,0
القضايا التجارية,3,48,0,1,1,1,2
القضايا الجنائية,0,0,52,1,0,1,0
القضايا العقارية,0,0,0,65,0,0,0
القضايا العمالية,1,2,0,0,89,0,0
القضايا المالية,2,2,0,1,0,43,4
قضايا الأحوال الشخصية,2,2,1,2,1,0,80


In [21]:
#: عرض الحالات اللي المودل غلط فيها  

import pandas as pd

# نحول التوقعات لسيريز ونربطها بنفس index حق y_test_text
y_pred_series = pd.Series(y_pred, index=y_test_text.index)

# نجيب الصفوف اللي فيها غلط
wrong_mask = y_pred_series != y_test_text

print("عدد الأخطاء:", wrong_mask.sum())

# نسوي جدول واضح للأخطاء
wrong_cases = pd.DataFrame({
    "النص": X_test_text[wrong_mask],
    "الحقيقة": y_test_text[wrong_mask],
    "توقع_المودل": y_pred_series[wrong_mask]
})

# نعرض أول 10 أخطاء
wrong_cases.head(10)


عدد الأخطاء: 38


,النص,الحقيقة,توقع_المودل
1534,منحي تصريح بناء وبعد بدء بناء توقفت بلديه تصري...,القضايا الإدارية,القضايا العقارية
1319,ورثه مكتب فصل منازعات اوراق تجاريه بعام والحكم...,قضايا الأحوال الشخصية,القضايا الإدارية
144,تغيير نشاط محل خياط الي حلاق رخصه,القضايا التجارية,القضايا الإدارية
1403,لمخالفات مروريه تسبب فيها طليقي سدد مبلغ والبا...,القضايا المالية,قضايا الأحوال الشخصية
1483,عاملتي منزليه هربت منزل واخذت بعض مجوهرات وقد ...,القضايا الجنائية,القضايا العقارية
1384,رفع شكوي محامي يماطل برفع طلب تماس دفع مبلغ,القضايا التجارية,القضايا المالية
1599,ام ارمله وتريد زواج واخوتي يرفضون ويعترضون يمك...,قضايا الأحوال الشخصية,القضايا العقارية
1549,وزاره تجاره صادرت بضاعتي محل بسبب انتهاء صلاحي...,القضايا الإدارية,القضايا التجارية
1160,توجيه انذار عملي بسبب منشور قديم وسايل تواصل م...,القضايا العمالية,القضايا الإدارية
1307,شخص عنده ارض ميراث ابوه وبعدما توفي ابوه قامو ...,قضايا الأحوال الشخصية,القضايا العقارية
